# Titanic Survival Prediction using Iguanas with Feature Engineering

This notebook demonstrates a complete end-to-end example of using **Iguanas** for rule-based classification on the Kaggle Titanic dataset, with advanced feature engineering using the **Gators** library.

The workflow includes:
1. Loading and exploring the data
2. **Feature engineering using Gators** (new columns, transformations, encoding)
3. Generating candidate rules using XGBoost
4. Filtering and selecting high-quality rules
5. Combining rules using different strategies
6. Generating predictions for submission

## 1. Import Libraries

In [14]:
import numpy as np
import polars as pl
from gators.data_cleaning import CastColumns, DropColumns, RenameColumns
from gators.discretizers import CustomDiscretizer
from gators.encoders import RareCategoryEncoder, WOEEncoder
from gators.feature_generation import ConditionFeatures, IsNull, MathFeatures, ScalarMathFeatures
from gators.feature_generation_str import (
    ExtractSubstring,
    Length,
    SplitExtract,
)
from gators.imputers import NumericImputer, StringImputer
from gators.pipeline import Pipeline
from xgboost import XGBClassifier

from iguanas.metrics import compute_metrics
from iguanas.rule_analysis import generate_rule_performance_report
from iguanas.rule_combination import (
    combine_rules_beam_search,
    combine_rules_cumulative,
    combine_rules_greedy,
)
from iguanas.rule_evaluation import apply_rules
from iguanas.rule_generation import rule_grid_search
from iguanas.rule_selection import filter_correlated_rules

## 2. Load and Prepare Data

Load the Titanic training data and separate features from the target variable (Survived).

In [2]:
train = pl.read_csv("../../../../../kaggle/titanic/train.csv").drop("PassengerId")
X_train = train.drop("Survived")
y_train = train["Survived"]

## 3. Feature Engineering with Gators

Build a comprehensive feature engineering pipeline using the **Gators** library. This pipeline will:
- Create missing value indicators for Age and Cabin
- Extract string features (name titles, cabin deck, ticket length)
- Calculate family size and fare per person
- Create categorical bins for age
- Generate feature interactions
- Apply Weight of Evidence (WOE) encoding for all categorical variables

**Key transformations in this pipeline:**

1. **Missing value handling**: Create indicators for missing Age/Cabin, then impute
2. **Feature extraction**: Extract passenger titles from names, cabin deck letters, ticket lengths
3. **Feature creation**: Calculate family size, fare per person, traveling alone indicator
4. **Discretization**: Convert continuous Age into categorical bins
5. **Interactions**: Create combinations of Pclass, Age, CabinDeck, and Embarked
6. **Encoding**: Apply WOE encoding to convert all categorical features to numeric values

In [15]:
# Define the feature engineering pipeline
steps = [
    # Create missing value indicators
    ("IsNull", IsNull(subset=["Age", "Cabin"])),
    # String feature engineering
    ("Length", Length(subset=["Ticket"])),
    ("SplitExtractName", SplitExtract(subset=["Name"], by=", ", n=1)),
    ("SplitExtractTitle", SplitExtract(subset=["Name__split_,__1"], by=".", n=0)),
    # Calculate family size (SibSp + Parch + 1)
    (
        "MathFeatures",
        MathFeatures(groups=[["SibSp", "Parch"]], func=["sum"], new_column_names=["Dummy"]),
    ),
    (
        "ScalarMathFeatures",
        ScalarMathFeatures(
            operations=[{"column": "Dummy_sum", "op": "+", "scalar": 1}],
            new_column_names=["FamilySize"],
        ),
    ),
    # Extract cabin deck (first letter of cabin)
    ("ExtractSubstring", ExtractSubstring(subset=["Cabin"], start=0, end=1)),
    # Rename for clarity
    (
        "RenameColumns",
        RenameColumns(
            column_mapping={
                "Name__split_,__1__split_._0": "Title",
                "Cabin__start0_end1": "CabinDeck",
            }
        ),
    ),
    # Handle rare categories (group infrequent values)
    ("RareCategoryEncoder", RareCategoryEncoder(min_count=0.01)),
    # Calculate fare per person
    (
        "MathFeatures2",
        MathFeatures(
            groups=[["Fare", "FamilySize"]], func=["div"], new_column_names=["FarePerPerson"]
        ),
    ),
    # Create 'traveling alone' indicator
    (
        "ConditionFeatures",
        ConditionFeatures(
            conditions=[{"column": "FamilySize", "op": ">", "value": 1}],
            new_column_names=["IsAlone"],
        ),
    ),
    # Drop raw columns no longer needed
    ("DropColumns", DropColumns(subset=["Cabin", "Ticket", "Dummy_sum"])),
    # Impute missing values
    ("NumericImputer", NumericImputer(strategy="mean")),
    ("StringImputer", StringImputer(strategy="constant", value="MISSING")),
    # as_numerics=True outputs float bin indices, required for ONNX export
    ("CustomDiscretizer", CustomDiscretizer(bins={"Age": [0, 12, 18, 35, 60, 100]}, inplace=True, as_numerics=True)),
    # Convert passenger class to categorical
    ("CastColumns", CastColumns(subset=["Pclass"], dtype=pl.String)),
    # Apply Weight of Evidence encoding (converts all categorical features to numeric)
    ("WOEEncoder", WOEEncoder()),
]

# Build and fit the pipeline
pipe = Pipeline(steps=steps, verbose=True)
X_train_transformed = pipe.fit_transform(X_train, y_train)

print(f"\nOriginal features: {X_train.shape[1]}")
print(f"Engineered features: {X_train_transformed.shape[1]}")

[Pipeline] fit+transform   1/17 · IsNull  |  in: rows=891  cols=10  nulls=866  →  out: rows=891  cols=12  nulls=866  (0.001s)
[Pipeline] fit+transform   2/17 · Length  |  in: rows=891  cols=12  nulls=866  →  out: rows=891  cols=13  nulls=866  (0.001s)
[Pipeline] fit+transform   3/17 · SplitExtractName  |  in: rows=891  cols=13  nulls=866  →  out: rows=891  cols=13  nulls=866  (0.001s)
[Pipeline] fit+transform   4/17 · SplitExtractTitle  |  in: rows=891  cols=13  nulls=866  →  out: rows=891  cols=13  nulls=866  (0.001s)
[Pipeline] fit+transform   5/17 · MathFeatures  |  in: rows=891  cols=13  nulls=866  →  out: rows=891  cols=14  nulls=866  (0.000s)
[Pipeline] fit+transform   6/17 · ScalarMathFeatures  |  in: rows=891  cols=14  nulls=866  →  out: rows=891  cols=15  nulls=866  (0.000s)
[Pipeline] fit+transform   7/17 · ExtractSubstring  |  in: rows=891  cols=15  nulls=866  →  out: rows=891  cols=16  nulls=1553  (0.000s)
[Pipeline] fit+transform   8/17 · RenameColumns  |  in: rows=891  co

## 4. Generate Candidate Rules

Use XGBoost-based grid search to generate candidate rules from the engineered features. The `rule_grid_search_parallel_scales` function trains models with different `scale_pos_weight` values and extracts rules from the decision trees.

In [16]:
estimator = XGBClassifier(n_estimators=100, max_depth=4, eval_metric="logloss", random_state=0)
rules = rule_grid_search(
    estimator, X_train_transformed, y_train, scale_pos_weights=np.logspace(0, 3, 50)
)

In [5]:
print(f"Number of rules generated: {len(rules)}")

Number of rules generated: 1971


## 5. Select High-Quality Rules

Apply the generated rules to the training data, compute performance metrics, and filter based on:
- Minimum precision (> 0.15)
- Minimum recall (> 0.15)
- Maximum correlation between rules (< 0.8)

This ensures we keep only the most useful and diverse rules.

In [17]:
R = apply_rules(X_train_transformed, rules.select("rule").to_series().to_list())
M = compute_metrics(R, y_train)
M = M.filter((pl.col("precision") > 0.15) & (pl.col("recall") > 0.15)).sort(
    "accuracy", descending=True
)
importance = dict(zip(M["rule"], M["f0.5"], strict=False))
uncorrelated_rules = filter_correlated_rules(
    R[M["rule"].to_list()], importance=importance, max_corr=0.8
)

In [7]:
num_rules = len(uncorrelated_rules)
print(f"Number of selected rules: {num_rules}")

Number of selected rules: 33


## 6. Combine Rules

Test different rule combination strategies to find the best performing ruleset.

### 6.1 Cumulative Combination

Combines rules cumulatively (rule1 OR rule2 OR ... OR ruleN):

In [8]:
R_combined = combine_rules_cumulative(
    R[uncorrelated_rules], output_names=[f"combined_rule_{i}" for i in range(1, num_rules + 1)]
)
M_combined = compute_metrics(R_combined, y_train).sort("accuracy", descending=True)
M_combined.head(3)

rule,TP,FP,TN,FN,precision,recall,accuracy,flagged(%),good_flagged(%),f0.25,f0.5,f1,f1.5,f2,num_rules
str,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""combined_rule_4""",252,57,492,90,0.815534,0.736842,0.835017,34.680135,10.382514,0.810443,0.798479,0.774194,0.759388,0.751342,1
"""combined_rule_5""",252,59,490,90,0.810289,0.736842,0.832772,34.904602,10.746812,0.805566,0.794451,0.771822,0.757982,0.750447,1
"""combined_rule_6""",252,59,490,90,0.810289,0.736842,0.832772,34.904602,10.746812,0.805566,0.794451,0.771822,0.757982,0.750447,1


### 6.2 Greedy Search

Uses a greedy algorithm to iteratively select the best rule combination:

In [9]:
R_greedy = combine_rules_greedy(R[uncorrelated_rules], y_train, metric="accuracy")
M_greedy = compute_metrics(R_greedy, y_train)
M_greedy

rule,TP,FP,TN,FN,precision,recall,accuracy,flagged(%),good_flagged(%),f0.25,f0.5,f1,f1.5,f2,num_rules
str,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""((X[""Title""] >= 0.25029) & (X[…",255,58,491,87,0.814696,0.745614,0.837262,35.129068,10.564663,0.81028,0.799875,0.778626,0.765589,0.758477,5


### 6.3 Beam Search

Uses beam search to explore rule combinations up to a maximum number of rules:

In [18]:
R_beam = combine_rules_beam_search(R[uncorrelated_rules], y_train, metric="accuracy", max_rules=10)
M_beam = compute_metrics(R_beam, y_train)
M_beam.head(3)

rule,TP,FP,TN,FN,precision,recall,accuracy,flagged(%),good_flagged(%),f0.25,f0.5,f1,f1.5,f2,mcc,num_rules
str,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""((X[""Title""] >= 0.25029) & (X[…",277,75,474,65,0.786932,0.809942,0.842873,39.506173,13.661202,0.788249,0.791429,0.798271,0.80272,0.805233,0.669825,5
"""((X[""Title""] >= 0.25029) & (X[…",277,75,474,65,0.786932,0.809942,0.842873,39.506173,13.661202,0.788249,0.791429,0.798271,0.80272,0.805233,0.669825,5
"""((X[""Title""] >= 0.25029) & (X[…",277,75,474,65,0.786932,0.809942,0.842873,39.506173,13.661202,0.788249,0.791429,0.798271,0.80272,0.805233,0.669825,5


## 7. Analyze the Best Ruleset

Generate a detailed report for the best performing ruleset from brute force combination:

In [11]:
for r in M_beam["rule"][0].split(" | "):
    print(r)

((X["Title"] >= 0.25029) & (X["FamilySize"] < 5.0))
((X["FarePerPerson_div"] >= 9.5) & (X["Sex"] >= 1.52977) & (X["Ticket__length"] >= 5.0) & (X["Fare"] < 151.55))
((X["Title"] >= 0.77539) & (X["Pclass"] >= 0.36447))
((X["Title"] >= 0.77539) & (X["Fare"] >= 31.3875) & (X["Fare"] < 151.55) & (X["Ticket__length"] < 7.0))


In [19]:
ruleset = M_beam["rule"][0]
print(f"Selected ruleset: {ruleset}")
report = generate_rule_performance_report(ruleset, X_train_transformed, y_train)
report

Selected ruleset: ((X["Title"] >= 0.25029) & (X["FamilySize"] < 5.0)) | ((X["CabinDeck"] >= 0.94252) & (X["Age"] < 5.0) & (X["Ticket__length"] < 10.0) & (X["Fare"] >= 7.775)) | ((X["Title"] >= 0.77539) & (X["FamilySize"] < 8.0) & (X["Pclass"] >= 0.36447)) | ((X["FarePerPerson_div"] >= 9.5) & (X["Sex"] >= 1.52977) & (X["Ticket__length"] >= 5.0) & (X["FamilySize"] < 4.0)) | ((X["Title"] >= 0.77539) & (X["Fare"] >= 31.3875) & (X["Fare"] < 151.55) & (X["Ticket__length"] < 7.0))


rule_index,rule,TP,FP,TN,FN,precision,recall,accuracy,flagged(%),good_flagged(%),f0.25,f0.5,f1,f1.5,f2,mcc,num_rules
str,str,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""0""","""((X[""Title""] >= 0.25029) & (X[…",277,75,474,65,0.786932,0.809942,0.842873,39.506173,13.661202,0.788249,0.791429,0.798271,0.80272,0.805233,0.669825,5
"""0.0""","""(X['Title'] >= 0.25029) & (X['…",239,57,492,103,0.807432,0.69883,0.820426,33.2211,10.382514,0.800118,0.783093,0.749216,0.729,0.718149,0.61435,1
"""0.1""","""(X['CabinDeck'] >= 0.94252) & …",86,17,532,256,0.834951,0.251462,0.693603,11.560045,3.096539,0.734673,0.570292,0.386517,0.320344,0.292318,0.335366,1
"""0.2""","""(X['Title'] >= 0.77539) & (X['…",166,9,540,176,0.948571,0.48538,0.792368,19.640853,1.639344,0.898154,0.796545,0.642166,0.571202,0.537913,0.574096,1
"""0.3""","""(X['FarePerPerson_div'] >= 9.5…",133,6,543,209,0.956835,0.388889,0.758698,15.600449,1.092896,0.881138,0.740535,0.553015,0.475784,0.441274,0.506557,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""0.3.3""","""(X['FamilySize'] < 4.0)""",311,489,60,31,0.38875,0.909357,0.416386,89.786756,89.071038,0.402298,0.439018,0.544658,0.643995,0.717251,0.029945,1
"""0.4.0""","""(X['Title'] >= 0.77539)""",249,98,451,93,0.717579,0.72807,0.785634,38.945006,17.850638,0.718188,0.719653,0.722787,0.72481,0.725948,0.548092,1
"""0.4.1""","""(X['Fare'] >= 31.3875)""",129,86,463,213,0.6,0.377193,0.664422,24.130191,15.664845,0.579852,0.536606,0.463196,0.425851,0.407454,0.25067,1


## 8. Generate Predictions on Test Data

Apply the same preprocessing pipeline to the test data, then use the best ruleset to generate predictions:

In [13]:
X_test = pl.read_csv("../../../../../kaggle/titanic/test.csv")
X_test_transformed = pipe.transform(X_test)
y_pred = eval(ruleset.replace("X", "X_test_transformed"))

[Pipeline] transform   1/17 · IsNull  |  in: rows=418  cols=11  nulls=414  →  out: rows=418  cols=13  nulls=414  (0.000s)
[Pipeline] transform   2/17 · Length  |  in: rows=418  cols=13  nulls=414  →  out: rows=418  cols=14  nulls=414  (0.000s)
[Pipeline] transform   3/17 · SplitExtractName  |  in: rows=418  cols=14  nulls=414  →  out: rows=418  cols=14  nulls=414  (0.001s)
[Pipeline] transform   4/17 · SplitExtractTitle  |  in: rows=418  cols=14  nulls=414  →  out: rows=418  cols=14  nulls=414  (0.001s)
[Pipeline] transform   5/17 · MathFeatures  |  in: rows=418  cols=14  nulls=414  →  out: rows=418  cols=15  nulls=414  (0.000s)
[Pipeline] transform   6/17 · ScalarMathFeatures  |  in: rows=418  cols=15  nulls=414  →  out: rows=418  cols=16  nulls=414  (0.000s)
[Pipeline] transform   7/17 · ExtractSubstring  |  in: rows=418  cols=16  nulls=414  →  out: rows=418  cols=17  nulls=741  (0.000s)
[Pipeline] transform   8/17 · RenameColumns  |  in: rows=418  cols=17  nulls=741  →  out: rows=41

In [ ]:
# Create submission file (Kaggle leaderboard score: 0.78)
# Note: +25% better than without feature engineering (0.60)
pl.DataFrame({"PassengerId": X_test["PassengerId"], "Survived": y_pred}).with_columns(
    pl.col("Survived").cast(pl.Int64)
).write_csv("submission_titanic.csv")

## 9. ONNX Export — Verify End-to-End Predictions Match

Build **two ONNX models** and stitch them into a single graph:

| Step | Tool | ONNX function |
|---|---|---|
| Feature engineering | Gators `Pipeline` | `pipeline_to_onnx` |
| Rule scoring | Iguanas ruleset | `rules_to_onnx` |
| End-to-end | combined | `pipeline_to_scoring_onnx` |

`pipeline_to_scoring_onnx` inserts a reshape bridge between the two models: it Unsqueezes each selected preprocessing output column `[N] → [N, 1]` and Concatenates them into the `[N, num_features]` matrix that the Iguanas rules model expects as its input `X`.

In [20]:
import onnxruntime as ort
from onnx import TensorProto
from gators.onnx_converters import pipeline_to_onnx, pipeline_to_scoring_onnx

from iguanas.onnx_converter import rules_to_onnx

# --- 1. Iguanas ruleset → ONNX ---
rules_onnx = rules_to_onnx(ruleset)

feature_map = {p.key: p.value for p in rules_onnx.metadata_props}
feature_cols = [feature_map[f"feature_{i}"] for i in range(len(feature_map))]
print(f"Rule features ({len(feature_cols)}): {feature_cols}")

# Ensure pipeline carries the fitted flag (fit_transform sets it from gators ≥ next release)
pipe._is_fitted = True
pipe._input_columns = list(X_train.columns)
pipe._input_dtypes = dict(zip(X_train.columns, X_train.dtypes))

# --- 2. Gators pipeline + Iguanas rules → single end-to-end ONNX model ---
#    pipeline_to_scoring_onnx bridges the preprocessing outputs into the
#    [N, num_features] matrix that rules_onnx expects as its input 'X'.
combined_onnx = pipeline_to_scoring_onnx(pipe, rules_onnx, feature_cols)
print(f"\nCombined ONNX inputs  : {[i.name for i in combined_onnx.graph.input]}")
print(f"Combined ONNX outputs : {[o.name for o in combined_onnx.graph.output]}")

Rule features (9): ['Title', 'FamilySize', 'CabinDeck', 'Age', 'Ticket__length', 'Fare', 'Pclass', 'FarePerPerson_div', 'Sex']


ValidationError: Field 'shape' of 'type' is required but missing.

In [ ]:
# --- 3. Build raw input feeds from X_train ---
#    Each pipeline input tensor is named '{col}__in'.
#    String dtypes → numpy object array; numerics → float32 (nulls become NaN).
feeds = {}
for inp in combined_onnx.graph.input:
    col = inp.name.removesuffix("__in")
    s = X_train[col]
    if inp.type.tensor_type.elem_type == TensorProto.STRING:
        feeds[inp.name] = s.fill_null("").to_numpy(allow_copy=True)
    else:
        feeds[inp.name] = s.to_numpy(allow_copy=True).astype("float32")

sess = ort.InferenceSession(combined_onnx.SerializeToString())
onnx_pred = sess.run(None, feeds)[0]  # int64 [N]

# --- 4. Reference predictions via Polars on already-transformed data ---
ref_pred = (
    apply_rules(X_train_transformed, [ruleset])[ruleset]
    .fill_null(False)
    .cast(pl.Int64)
    .to_numpy()
)

match = (onnx_pred == ref_pred).all()
print(f"Predictions match : {match}")
print(f"ONNX   positives  : {int(onnx_pred.sum())} / {len(onnx_pred)}")
print(f"Polars positives  : {int(ref_pred.sum())} / {len(ref_pred)}")
assert match, "ONNX and Polars predictions differ!"